In [1]:
import numpy as np
import matplotlib.pyplot as plt
from astropy.io import fits
from astropy.wcs import WCS
from astropy.coordinates import SkyCoord, ICRS, Galactic
import astropy.units as u
import matplotlib as mpl
from reproject import reproject_from_healpix, reproject_to_healpix
import healpy

## Stokes I

In [2]:
hduC = fits.open('/srv/data/chime/chime_QU_Oct2023_400_729/I_400_729_Oct2023_new_gal.fits')
hduD = fits.open('/srv/data/dragons/Mar2024_500_1000_MHz_cube/DRAGONS_500_to_1024_i_cube_32bit_gal.fits')

hdrC = hduC[0].header
hdrD = hduD[0].header

freqC = WCS(hdrC).all_pix2world(0,0,range(WCS(hdrC).array_shape[0]),0)[2]/1e6
freqD = np.genfromtxt('/srv/data/dragons/Mar2024_500_1000_MHz_cube/ListOfFrequencies_Hz.txt')/1e6

print(freqC[0],freqC[-1])
print(freqD[0],freqD[-1])

729.6875 400.390625
512.6875 1023.1875


In [3]:
def make_2D_hdr(hdr):

    hdr2D = hdr.copy()

    hdr2D['NAXIS'] = 2
    del hdr2D['NAXIS3']
    del hdr2D['CTYPE3']
    del hdr2D['CRVAL3']
    del hdr2D['CDELT3']

    try:
        del hdr2D['CRPIX3']
        del hdr2D['CUNIT3']
    except:
        pass

    return hdr2D
    

In [ ]:
fidxC = abs(freqC - 610).argmin()
fidxD = abs(freqD - 610).argmin()

print(freqC[fidxC])
print(freqD[fidxD])

sliceC = hduC[0].data[fidxC]
sliceD = hduD[0].data[fidxD]

hdr2D_C = make_2D_hdr(hdrC)
hdr2D_D = make_2D_hdr(hdrD)

#print(repr(hdr2D_C))
#print(repr(hdr2D_D))

fits.writeto('/srv/data/gmims/gmims-lbn/maps_for_Tom/chime_StokesI_610_gal.fits', sliceC.astype('float32'), hdr2D_C, overwrite=True)
fits.writeto('/srv/data/gmims/gmims-lbn/maps_for_Tom/dragons_StokesI_610_gal.fits', sliceD.astype('float32'), hdr2D_D, overwrite=True)

## Moments

In [4]:
hduC  = fits.open('/srv/data/chime/chime_FD_May2024_400_729/FDF_clean_tot.fits')
hduD  = fits.open('/srv/data/dragons/Mar2024_500_1000_MHz_cube/fd_cube_gal/FDF_clean_tot.fits')
hduCD = fits.open('/srv/data/gmims/gmims-lbn/experiments/results_filtered_highpass_Sep26/RM_synth/dragons_chime/FDF_clean_tot.fits')

hdrC = hduC[0].header
hdrD = hduD[0].header
hdrCD = hduCD[0].header

cubeC  = hduC[0].data
cubeD  = hduD[0].data
cubeCD = hduCD[0].data

print(cubeC.shape)
print(cubeD.shape)
print(cubeCD.shape)

FD_C  = WCS(hdrC).all_pix2world(0,0,range(WCS(hdrC).array_shape[0]),0)[2]
FD_D  = WCS(hdrD).all_pix2world(0,0,range(WCS(hdrD).array_shape[0]),0)[2]
FD_CD = WCS(hdrCD).all_pix2world(0,0,range(WCS(hdrCD).array_shape[0]),0)[2]


(801, 1440, 2880)
(801, 180, 360)
(601, 1440, 2880)


a floating-point value was expected. [astropy.wcs.wcs]


In [5]:
def FD_moments(data,peakthresh,maxFD,RM_arr,M0=True,M1=True,M2=False,*args,**kwargs):
 
    dFD = abs(RM_arr[1]-RM_arr[0])

    # cut out FD range chosen:
    data_use = data[(abs(RM_arr) <= maxFD),:,:]    
    RM_arr_use = RM_arr[(abs(RM_arr) <= maxFD)]  

    # set any data points below PI threshold to NaN:
    data_use[data_use < peakthresh] = np.nan

    moments = {}
    if M0:
        M0_data = dFD*np.nansum(data_use,axis=0)
        M0_data[M0_data == 0] = np.nan
        moments['M0'] = M0_data
    if M1:
        M1_data = dFD*np.nansum(data_use*RM_arr_use[:,np.newaxis,np.newaxis],axis=0)/M0_data
        moments['M1'] = M1_data
    if M2:
        M2_data = np.sqrt(dFD*np.nansum(data_use*(RM_arr_use[:,np.newaxis,np.newaxis]-M1_data)**2,axis=0)/M0_data)
        moments['M2'] = M2_data
        
    return(moments)

In [12]:
#M_C  = FD_moments(cubeC, 0.0,80,FD_C, M0=True,M1=True,M2=True)
M_D  = FD_moments(cubeD, 0.0,50,FD_D, M0=True,M1=True,M2=True)
#M_CD = FD_moments(cubeCD,0.0,80,FD_CD,M0=True,M1=True,M2=True)

In [19]:
hdr2D_C  = make_2D_hdr(hdrC)
hdr2D_D  = make_2D_hdr(hdrD)
hdr2D_CD = make_2D_hdr(hdrCD)

#print(repr(hdr2D_C))
#print('')
#print(repr(hdr2D_D))
#print('')
print(repr(hdr2D_CD))


SIMPLE  =                    T / conforms to FITS standard                      
BITPIX  =                  -32 / array data type                                
NAXIS   =                    2 / number of array dimensions                     
NAXIS1  =                 2880                                                  
NAXIS2  =                 1440                                                  
CRVAL1  =                180.0                                                  
CRVAL2  =                  0.0                                                  
CDELT1  =               -0.125                                                  
CDELT2  =                0.125                                                  
CRPIX1  =               1440.0                                                  
CRPIX2  =                720.0                                                  
CUNIT1  = 'deg     '                                                            
CUNIT2  = 'deg     '        

In [ ]:
M1_CD_hpx, footprint = reproject_to_healpix((M_CD['M1'],WCS(hdr2D_CD)),coord_system_out='C',nside=512)
M0_CD_hpx, footprint = reproject_to_healpix((M_CD['M0'],WCS(hdr2D_CD)),coord_system_out='C',nside=512)

M1_CD_hpx[np.isnan(M1_CD_hpx)] = healpy.UNSEEN
M0_CD_hpx[np.isnan(M0_CD_hpx)] = healpy.UNSEEN

r = healpy.Rotator(coord=['C', 'G'])
M1_CD_hpx_gal = r.rotate_map_alms(M1_CD_hpx)
M0_CD_hpx_gal = r.rotate_map_alms(M0_CD_hpx)

# Just use the CHIME-only galactic header:
M1_CD_gal, footprint = reproject_from_healpix((M1_CD_hpx_gal,'galactic'),WCS(hdr2D_C), nested=False)
M0_CD_gal, footprint = reproject_from_healpix((M0_CD_hpx_gal,'galactic'),WCS(hdr2D_C), nested=False)

In [ ]:
fits.writeto('/srv/data/gmims/gmims-lbn/maps_for_Tom/chime_M1_gal.fits',
             M_C['M1'].astype('float32'), hdr2D_C, overwrite=True)

fits.writeto('/srv/data/gmims/gmims-lbn/maps_for_Tom/dragons_M1_gal.fits',
             M_D['M1'].astype('float32'), hdr2D_D, overwrite=True)

fits.writeto('/srv/data/gmims/gmims-lbn/maps_for_Tom/chime_dragons_M1_gal.fits',
             M1_CD_gal.astype('float32'), hdr2D_C, overwrite=True)

In [ ]:
fits.writeto('/srv/data/gmims/gmims-lbn/maps_for_Tom/chime_M0_gal.fits',
             M_C['M0'].astype('float32'), hdr2D_C, overwrite=True)

fits.writeto('/srv/data/gmims/gmims-lbn/maps_for_Tom/dragons_M0_gal.fits',
             M_D['M0'].astype('float32'), hdr2D_D, overwrite=True)

fits.writeto('/srv/data/gmims/gmims-lbn/maps_for_Tom/chime_dragons_M0_gal.fits',
             M0_CD_gal.astype('float32'), hdr2D_C, overwrite=True)

In [20]:
fits.writeto('/srv/data/gmims/gmims-lbn/maps_for_Tom/dragons_M2_gal.fits',
             M_D['M2'].astype('float32'), hdr2D_D, overwrite=True)

## Peaks

In [ ]:
hduC  = fits.open('/srv/data/chime/chime_FD_May2024_400_729/dirty_phiPeakPIfit_rm2.fits')
hduD  = fits.open('/srv/data/dragons/Mar2024_500_1000_MHz_cube/fd_cube_gal/FDF_peakRM.fits')
hduCD = fits.open('/srv/data/gmims/gmims-lbn/experiments/results_filtered_highpass_Sep26/RM_synth/dragons_chime/clean_phiPeakPIfit_rm2.fits')

hdrC  = hduC[0].header
hdrD  = hduD[0].header
hdrCD = hduCD[0].header

print(hduC[0].data.shape)
print(hduD[0].data.shape)
print(hduCD[0].data.shape)

#print(repr(hdrC))
#print('')
#print(repr(hdrD))
#print('')
#print(repr(hdrCD))

In [ ]:
hdr2D_D = make_2D_hdr(hdrD)
print(repr(hdr2D_D))

In [ ]:
peak_CD_hpx, footprint = reproject_to_healpix((hduCD[0].data,WCS(hdrCD)),coord_system_out='C',nside=512)

peak_CD_hpx[np.isnan(peak_CD_hpx)] = healpy.UNSEEN

r = healpy.Rotator(coord=['C', 'G'])
peak_CD_hpx_gal = r.rotate_map_alms(peak_CD_hpx)

# Just use the CHIME-only galactic header:
peak_CD_gal, footprint = reproject_from_healpix((peak_CD_hpx_gal,'galactic'),WCS(hdrC), nested=False)

In [ ]:
fits.writeto('/srv/data/gmims/gmims-lbn/maps_for_Tom/chime_peakFD_gal.fits',
             hduC[0].data.astype('float32'), hdrC, overwrite=True)

fits.writeto('/srv/data/gmims/gmims-lbn/maps_for_Tom/dragons_peakFD_gal.fits',
             hduD[0].data[0].astype('float32'), hdr2D_D, overwrite=True)

fits.writeto('/srv/data/gmims/gmims-lbn/maps_for_Tom/chime_dragons_peakFD_gal.fits',
             peak_CD_gal.astype('float32'), hdrC, overwrite=True)

## Rotate existing files to Equatorial

In [ ]:
def make_hdr_eq(hdr):

    hdr_eq = hdr.copy()

    hdr_eq['CTYPE1'] = 'RA---CAR'
    hdr_eq['CTYPE2'] = 'DEC--CAR'
    del hdr_eq['COORDSYS']

    #print(repr(hdr_eq))

    return hdr_eq

In [ ]:
filename = 'chime_M1_gal.fits'
nside = 512

hdu = fits.open('/srv/data/gmims/gmims-lbn/maps_for_Tom/'+filename)
hdr = hdu[0].header

hdr_eq = make_hdr_eq(hdr)

map_hpx, footprint = reproject_to_healpix((hdu[0].data,WCS(hdr)),coord_system_out='G',nside=nside)

map_hpx[np.isnan(map_hpx)] = healpy.UNSEEN

r = healpy.Rotator(coord=['G', 'C'])
map_hpx_eq = r.rotate_map_alms(map_hpx)

map_eq, footprint = reproject_from_healpix((map_hpx_eq,'ICRS'),WCS(hdr_eq), nested=False)


## Tadpole tests

In [ ]:
hdu = fits.open('/srv/data/chime/tadpole_cutout/RMsynth_400_729/QU/clean_200mJy/FDF_clean_tot.fits')
data = hdu[0].data
hdr = hdu[0].header
FD  = WCS(hdr).all_pix2world(0,0,range(WCS(hdr).array_shape[0]),0)[2]

print(data.shape)
print(FD)

In [ ]:
M  = FD_moments(data, 0.0,200,FD, M0=True,M1=True,M2=False)

In [ ]:
plt.imshow(M['M0'],origin='lower',vmin=0,vmax=350)